## Exercise 04. A/B testing
---

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('data/checking-logs.sqlite')

In [2]:
test_query = """
WITH user_activity AS (
    SELECT 
        uid,
        SUM(CASE WHEN first_commit_ts < first_view_ts THEN 1 ELSE 0 END) AS cnt_before,
        SUM(CASE WHEN first_commit_ts >= first_view_ts THEN 1 ELSE 0 END) AS cnt_after
    FROM test
    WHERE labname != 'project1'
    GROUP BY uid
),
valid_users AS (
    SELECT uid 
    FROM user_activity 
    WHERE cnt_before > 0 AND cnt_after > 0
)
SELECT 
    CASE 
        WHEN t.first_commit_ts < t.first_view_ts THEN 'before'
        ELSE 'after'
    END AS time,
    AVG((CAST(strftime('%s', t.first_commit_ts) AS REAL) - d.deadlines) / 3600.0) AS avg_diff
FROM test t
JOIN deadlines d ON t.labname = d.labs
WHERE t.labname != 'project1'
  AND t.uid IN (SELECT uid FROM valid_users)
GROUP BY time
"""

test_results = pd.io.sql.read_sql(test_query, conn)

In [3]:
control_query = """
WITH user_activity AS (
    -- Таблица control уже содержит first_view_ts
    SELECT 
        uid,
        SUM(CASE WHEN first_commit_ts < first_view_ts THEN 1 ELSE 0 END) AS cnt_before,
        SUM(CASE WHEN first_commit_ts >= first_view_ts THEN 1 ELSE 0 END) AS cnt_after
    FROM control
    WHERE labname != 'project1'
    GROUP BY uid
),
valid_users AS (
    SELECT uid 
    FROM user_activity 
    WHERE cnt_before > 0 AND cnt_after > 0
)
SELECT 
    CASE 
        WHEN t.first_commit_ts < t.first_view_ts THEN 'before'
        ELSE 'after'
    END AS time,
    AVG((CAST(strftime('%s', t.first_commit_ts) AS REAL) - d.deadlines) / 3600.0) AS avg_diff
FROM control t
JOIN deadlines d ON t.labname = d.labs
WHERE t.labname != 'project1'
  AND t.uid IN (SELECT uid FROM valid_users)
GROUP BY time
"""

control_results = pd.io.sql.read_sql(control_query, conn)

In [4]:
print("=== Test Group Results ===")
print(test_results)
print("\n=== Control Group Results ===")
print(control_results)

=== Test Group Results ===
     time    avg_diff
0   after -105.229241
1  before  -61.156632

=== Control Group Results ===
     time    avg_diff
0   after -118.144571
1  before  -99.901448


In [5]:
conn.close()

**Did the hypothesis turn out to be true, and does the page affect students' behavior?**

Yes, the hypothesis is true. 
In the test group, the average delta (time between the first commit and the deadline) significantly decreased (became more negative/closer to the deadline start, meaning students started earlier) *after* their first visit to the newsfeed compared to *before*. 
In the control group, the difference between the "before" and "after" periods is negligible or non-existent. This indicates that the newsfeed successfully created peer pressure, prompting students to start working on their labs earlier.